In [32]:
import os
import os
import sqlite3
import datetime
app_name = "e2canavipuntos"
base_path = os.path.join(os.environ["LOCALAPPDATA"], app_name)
os.makedirs(base_path, exist_ok=True)

db_path = os.path.join(base_path, f"{app_name}.db")
print(f"Base de datos en: {db_path}")

def conectar_db(Path =db_path):
    return sqlite3.connect(db_path)

Base de datos en: C:\Users\nicol\AppData\Local\e2canavipuntos\e2canavipuntos.db


In [13]:
def tabla_existe(conn, nombre_tabla):
    cursor = conn.cursor()
    cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type='table' AND name=?
    """, (nombre_tabla,))
    return cursor.fetchone() is not None


def crear_tabla(conn, nombre_tabla, columnas, tipos):
    if tabla_existe(conn, nombre_tabla):
        print(f"La tabla '{nombre_tabla}' ya existe.")
        return
    else:
        cursor = conn.cursor()
        columnas_definicion = ", ".join(f"{col} {tipo}" for col, tipo in zip(columnas, tipos))
        cursor.execute(f"CREATE TABLE {nombre_tabla} ({columnas_definicion})")
        conn.commit()
        print(f"Tabla '{nombre_tabla}' creada correctamente.")



def insertar_dato(conn, nombre_tabla, columnas, valores):
    cursor = conn.cursor()
    placeholders = ", ".join("?" * len(valores))
    cursor.execute(f"INSERT INTO {nombre_tabla} ({', '.join(columnas)}) VALUES ({placeholders})", valores)
    conn.commit()


def consultar_usuario(conn, nombre_tabla, columnas, valores):
    cursor = conn.cursor()
    placeholders = ", ".join("?" * len(valores))
    cursor.execute(f"SELECT {', '.join(columnas)} FROM {nombre_tabla} WHERE {', '.join(f'{col} = ?' for col in columnas)}", valores)
    return cursor.fetchall()

def actualizar_usuario(conn, uid, nuevo_nombre=None, nuevo_activo=None):
    cursor = conn.cursor()

    if nuevo_nombre is not None and nuevo_activo is not None:
        cursor.execute("""
            UPDATE usuarios
            SET nombre = ?, activo = ?
            WHERE uid = ?
        """, (nuevo_nombre, nuevo_activo, uid))

    elif nuevo_nombre is not None:
        cursor.execute("""
            UPDATE usuarios
            SET nombre = ?
            WHERE uid = ?
        """, (nuevo_nombre, uid))

    elif nuevo_activo is not None:
        cursor.execute("""
            UPDATE usuarios
            SET activo = ?
            WHERE uid = ?
        """, (nuevo_activo, uid))

    else:
        print("No se enviaron cambios para actualizar.")
        return

    conn.commit()
    print(f"Usuario con UID {uid} actualizado.")


def main():
    conn = conectar_db()

    #crear_tabla_usuarios(conn)

    if tabla_existe(conn, "usuarios"):
        print("Validación OK: la tabla existe.")

    conn.close()




In [25]:
conn = conectar_db(db_path)

crear_tabla(conn, "Usuarios", ["IdUser", "Nombres", "Apellidos", "Correo", "Documento", "FechaNacimiento", "Celular", "Eps"], ["TEXT PRIMARY KEY", "TEXT", "TEXT", "TEXT", "INTEGER", "TEXT", "INTEGER", "TEXT", "INTEGER", "TEXT"])
crear_tabla(conn, "Puntos", ["IdTrPuntos", "IdTr", "IdUser", "Cantidad", "FechaInicio", "FechaFin"], ["TEXT PRIMARY KEY", "TEXT", "TEXT", "INTEGER", "TEXT", "TEXT"])
crear_tabla(conn, "Membresias", ["IdMembresia", "IdUser", "IdTr", "Membresia", "Descuento", "FechaInicio", "FechaFin"], ["TEXT PRIMARY KEY", "TEXT", "TEXT", "TEXT", "REAL", "TEXT", "TEXT"])
crear_tabla(conn, "Transacciones", ["IdTr", "IdUser", "Monto", "Fecha"], ["TEXT PRIMARY KEY", "TEXT", "INTEGER", "TEXT"])

La tabla 'Usuarios' ya existe.
Tabla 'Puntos' creada correctamente.
La tabla 'Membresias' ya existe.
La tabla 'Transacciones' ya existe.


In [28]:
def Agregar_usuario(conn, uid, nombres, apellidos, correo, documento, fecha_nacimiento, celular, eps):
    try:
        insertar_dato(conn, "Usuarios", ["IdUser", "Nombres", "Apellidos", "Correo", "Documento", "FechaNacimiento", "Celular", "Eps"], [uid, nombres, apellidos, correo, documento, fecha_nacimiento, celular, eps])
        print(f"Usuario '{nombres} {apellidos}' agregado correctamente.")
    except Exception as e:
        print(f"Error al insertar usuario: {e}")

def Agregar_Membresia(conn, id_membresia, id_user, id_tr, membresia, descuento, fecha_inicio, fecha_fin):
    try:
        insertar_dato(conn, "Membresias", ["IdMembresia", "IdUser", "IdTr", "Membresia", "Descuento", "FechaInicio", "FechaFin"], [id_membresia, id_user, id_tr, membresia, descuento, fecha_inicio, fecha_fin])
        print(f"Membresía '{membresia}' para usuario '{id_user}' agregada correctamente.")
    except Exception as e:
        print(f"Error al insertar membresía: {e}")

def Agregar_Puntos(conn, id_tr_puntos, id_tr, id_user, cantidad, fecha_inicio, fecha_fin):
    try:
        insertar_dato(conn, "Puntos", ["IdTrPuntos", "IdTr", "IdUser", "Cantidad", "FechaInicio", "FechaFin"], [id_tr_puntos, id_tr, id_user, cantidad, fecha_inicio, fecha_fin])
        print(f"Puntos para usuario '{id_user}' agregados correctamente.")
    except Exception as e:
        print(f"Error al insertar puntos: {e}")

def Agregar_Transaccion(conn, id_tr, id_user, monto, fecha):
    try:
        insertar_dato(conn, "Transacciones", ["IdTr", "IdUser", "Monto", "Fecha"], [id_tr, id_user, monto, fecha])
        print(f"Transacción para usuario '{id_user}' agregada correctamente.")  
    except Exception as e:
        print(f"Error al insertar transacción: {e}")



In [ ]:
Agregar_usuario(conn, "US01", "Yisus", "Castelblanco", "soydios@gmail.com", "1", "24/12/-34", "14818114", "Templarios Anonimos")
Agregar_Membresia(conn, "MB01", "US01", "TR01", "Catolicismo", "1", "0/0/0", "31/12/9999")
Agregar_Puntos(conn, "PT01", "TR01", "US01", 666, "0/0/0", "31/12/5000")
Agregar_Transaccion(conn, "TR01", "US01", 0, "0/0/0")


Error al insertar usuario: UNIQUE constraint failed: Usuarios.IdUser
Error al insertar membresía: UNIQUE constraint failed: Membresias.IdMembresia
Error al insertar puntos: UNIQUE constraint failed: Puntos.IdTrPuntos
Error al insertar transacción: UNIQUE constraint failed: Transacciones.IdTr


In [ ]:
def Agregar_UsuarioFull(conn, Nombres, Apellidos, Correo, Documento, FechaNacimiento, Celular, Eps):
    uid = f"US{Documento}"
    Agregar_usuario(conn, uid, Nombres, Apellidos, Correo, Documento, FechaNacimiento, Celular, Eps)

def Agregar_MembresiaFull(conn, IdUser, IdTr, Membresia, Descuento, FechaInicio, FechaFin):
    now = datetime.now()
    FechaExacta = now.strftime("%d%m%Y%H%M%S")
    id_membresia = f"MB{FechaExacta}"
    Agregar_Membresia(conn, id_membresia, IdUser, IdTr, Membresia, Descuento, FechaInicio, FechaFin) 

def Agregar_PuntosFull(conn, IdUser, Cantidad):
    now = datetime.now()
    FechaExacta = now.strftime("%d%m%Y%H%M%S")
    id_tr_puntos = f"PT{FechaExacta}"
    id_tr = f"TR{FechaExacta}"

    FechaInicio = now.strftime("%d/%m/%Y")
    FechaFin = now.addedelta(days=365).strftime("%d/%m/%Y").strftime("%d/%m/%Y")
    Agregar_Puntos(conn, id_tr_puntos, id_tr, IdUser, Cantidad, FechaInicio, FechaFin)   
    Agregar_Transaccion(conn, id_tr, IdUser, 0, FechaInicio)

def Agregar_TransaccionFull(conn, IdUser, Monto):
    now = datetime.now()
    FechaExacta = now.strftime("%d%m%Y%H%M%S")
    id_tr = f"TR{FechaExacta}"
    Fecha = now.strftime("%d/%m/%Y")
    Agregar_Transaccion(conn, id_tr, IdUser, Monto, Fecha)

def Agregar_TransaccionPuntosFull(conn, IdUser, Monto):
    now = datetime.now()
    FechaExacta = now.strftime("%d%m%Y%H%M%S")
    id_tr = f"TR{FechaExacta}"
    Fecha = now.strftime("%d/%m/%Y")
    Agregar_Transaccion(conn, id_tr, IdUser, Monto, Fecha)
    cantidad_puntos = Monto // 1000
    FechaFin = now.addedelta(days=365).strftime("%d/%m/%Y")
    Agregar_Puntos(conn, f"PT{FechaExacta}", id_tr, IdUser, cantidad_puntos, Fecha, FechaFin)

## Prueba del lector ACR122U (PC/SC)

Esta prueba usa el controlador PC/SC de Windows, que es el modo normal del ACR122U. Si aparece `ModuleNotFoundError: smartcard`, instala primero `pyscard` en el mismo entorno del notebook: `python -m pip install pyscard`.

In [ ]:
from smartcard.System import readers
from smartcard.Exceptions import CardConnectionException, NoCardException

def probar_acr122u():
    """Lista los lectores PC/SC y lee el UID de una tarjeta en el ACR122U."""
    lectores = list(readers())
    print("Lectores PC/SC detectados:")
    for lector in lectores:
        print(f"  - {lector}")

    if not lectores:
        print("\nNo se detectó ningún lector. Conecta el ACR122U y verifica el controlador PC/SC/CCID de Windows.")
        return None

    lector = next((item for item in lectores if "ACR122" in str(item).upper()), lectores[0])
    conexion = lector.createConnection()

    try:
        print(f"\nUsando: {lector}")
        print("Acerca una tarjeta NFC al lector…")
        conexion.connect()

        # APDU PC/SC para recuperar el UID de tarjetas ISO 14443.
        datos, sw1, sw2 = conexion.transmit([0xFF, 0xCA, 0x00, 0x00, 0x00])
        if (sw1, sw2) != (0x90, 0x00):
            print(f"El lector respondió {sw1:02X} {sw2:02X}; no pudo obtener el UID.")
            return None

        uid = "".join(f"{byte:02X}" for byte in datos)
        print(f"UID leído correctamente: {uid}")
        return uid
    except NoCardException:
        print("No hay tarjeta sobre el lector. Acércala e intenta de nuevo.")
    except CardConnectionException as error:
        print(f"No fue posible conectar con el lector: {error}")
    finally:
        try:
            conexion.disconnect()
        except Exception:
            pass

probar_acr122u()

In [24]:
conn.close()